# 03. Seed stability and tuning summary

## Setup

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # model_aza

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from model_aza.training import tuning
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## Inputs

In [ ]:
spatial_run = ""  # Optional pipeline_outputs/tuning/spatial_<id>; blank uses the latest spatial run
n_estimators = 1000
seeds = (1, 2, 3, 4, 5)
spatial_run = Path(spatial_run) if spatial_run else tuning.latest_run(root, "spatial")
spatial_inputs = pd.read_json(spatial_run / "inputs.json", typ="series")
features_path, development = tuning.load_development(root, spatial_inputs["development_features"])
spatial_records = tuning.load_records(spatial_run, "spatial_cv")
print("Spatial run:", spatial_run)
print("Development features:", features_path)

## Candidate settings

In [ ]:
candidates = []
for cohort in tuning.COHORTS:
    rows = [r for r in spatial_records if r["cohort"] == cohort]
    best = max(rows, key=lambda r: r["spatial_macro_f1"])
    baseline = max((r for r in rows if r["baseline_setting"]), key=lambda r: r["validation_macro_f1"])
    for role, record in (("baseline", baseline), ("tuned", best)):
        candidates.append({"cohort": cohort, "role": role,
                           "params": {**{k: record[k] for k in tuning.PARAMETERS}, "n_estimators": n_estimators}})
pd.DataFrame([{"cohort": c["cohort"], "role": c["role"], **c["params"]} for c in candidates])

## Run seed repeats

In [ ]:
run = tuning.new_run(root, "stability", features_path, spatial_run=str(spatial_run),
                     n_estimators=n_estimators, seeds=list(seeds))
records = []
for candidate in candidates:
    for record in tuning.seed_stability(development, candidate["cohort"], candidate["params"], seeds):
        records.append({"role": candidate["role"], **record})
    print(candidate["cohort"], candidate["role"], "done", flush=True)
print("Saved:", tuning.save_records(run, "seed_stability", records))

## Seed stability table

In [ ]:
stability = pd.DataFrame(records)
metrics = ["macro_f1", "balanced_accuracy", "carrot_f1", "cabbage_f1", "potato_f1"]
summary = stability.groupby(["cohort", "role"])[metrics].agg(["mean", "std"]).round(3)
display(summary)

## Summary table

In [ ]:
settings = pd.DataFrame([{"cohort": c["cohort"], "role": c["role"], **c["params"]} for c in candidates])
spatial = pd.DataFrame(spatial_records)
keys = ["cohort", "max_features", "min_samples_leaf", "max_samples", "max_depth"]
for frame in (settings, spatial):
    frame["max_depth"] = frame["max_depth"].astype(str)
    frame["max_samples"] = frame["max_samples"].astype(str)
overview = (settings.merge(spatial[keys + ["validation_macro_f1", "spatial_macro_f1"]], on=keys, how="left")
            .merge(stability.groupby(["cohort", "role"]).macro_f1.agg(seed_mean="mean", seed_std="std").reset_index(),
                   on=["cohort", "role"]))
overview.to_csv(run / "tuning_summary.csv", index=False)
display(overview.round(3))

## Figure

In [ ]:
figure, axis = plt.subplots(figsize=(9, 4))
table = stability.pivot_table(index="cohort", columns="role", values="macro_f1", aggfunc=["mean", "std"])
table["mean"].plot(kind="bar", yerr=table["std"], ax=axis, capsize=4)
axis.set_ylabel("Validation parcel macro-F1 (mean ± sd over seeds)")
axis.set_ylim(0, 1)
plt.tight_layout()
figure.savefig(run / "seed_stability.png", dpi=150)
display(figure)
plt.close(figure)
print("Run folder:", run)